# CUSTOMER SEGMENTATION USING CLUSTERING
## Academic Machine Learning Project - Unsupervised Learning Pipeline

### Project Overview
This notebook demonstrates a complete, college-level unsupervised machine learning workflow to group customers into distinct behavioral segments based on demographic attributes and purchasing metrics.

---

### 1. Import Required Libraries

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Add parent directory to access src modules
sys.path.append(os.path.abspath('..'))

from src.preprocessing import DataPreprocessor
from src.eda import EDAnalyzer
from src.clustering import ClusteringPipeline
from src.evaluation import ModelEvaluator
from src.segmentation import SegmentAnalyzer

print('All modules loaded successfully!')

### 2. Load & Inspect Dataset

In [ ]:
data_path = '../data/customers.csv'
preprocessor = DataPreprocessor()
df_raw = preprocessor.load_data(data_path)

print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
df_raw.head()

### 3. Data Preprocessing & Quality Checks

In [ ]:
# Data Quality Report
quality_report = preprocessor.check_data_quality(df_raw)
print("Missing Values:", quality_report['missing_values'])
print("Duplicates:", quality_report['duplicate_count'])

# Clean Data & Encode Categorical Variables
df_clean = preprocessor.clean_data(df_raw)
df_encoded = preprocessor.encode_categorical(df_clean)

# Feature Selection & Scaling
feature_cols = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)', 'Purchase Frequency (per year)']
X_scaled, df_ready, scaler = preprocessor.prepare_features(df_encoded, feature_cols)

print(f"Scaled Feature Matrix Shape: {X_scaled.shape}")

### 4. Exploratory Data Analysis (EDA)

In [ ]:
eda = EDAnalyzer(output_dir='../outputs/plots')

# Distributions
eda.plot_distributions(df_clean, save=False)
plt.show()

# Gender Distribution
eda.plot_gender_distribution(df_clean, save=False)
plt.show()

# Bivariate Scatter Plots
eda.plot_bivariate_relationships(df_clean, save=False)
plt.show()

# Heatmap
eda.plot_correlation_heatmap(df_clean, save=False)
plt.show()

### 5. K-Means Clustering & Hyperparameter Tuning
We evaluate K-Means across $K \in [2, 10]$ using the **Elbow Method (WCSS)** and **Silhouette Score**.

In [ ]:
pipeline = ClusteringPipeline(random_state=42)
search_res = pipeline.find_optimal_kmeans(X_scaled, min_k=2, max_k=10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# WCSS Elbow Curve
ax1.plot(search_res['k_values'], search_res['inertias'], 'bo-', linewidth=2, markersize=8)
ax1.set_title('Elbow Method (WCSS vs K)', fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia / WCSS')

# Silhouette Score Curve
ax2.plot(search_res['k_values'], search_res['silhouette_scores'], 'ro-', linewidth=2, markersize=8)
ax2.set_title('Silhouette Score vs K', fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

best_k = search_res['k_values'][np.argmax(search_res['silhouette_scores'])]
print(f"Selected Optimal K: {best_k}")

In [ ]:
kmeans_labels, kmeans_model, centers = pipeline.fit_kmeans(X_scaled, n_clusters=best_k)

### 6. Hierarchical Agglomerative Clustering & Dendrogram

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

link_matrix = linkage(X_scaled, method='ward')
plt.figure(figsize=(12, 6))
dendrogram(link_matrix, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=10)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)', fontweight='bold')
plt.xlabel('Customer Sample Index / Cluster Size')
plt.ylabel('Euclidean Distance')
plt.show()

hier_labels, hier_model = pipeline.fit_hierarchical(X_scaled, n_clusters=best_k)

### 7. DBSCAN Clustering (Density-Based Spatial Clustering)

In [ ]:
dbscan_labels, dbscan_model, noise_cnt = pipeline.fit_dbscan(X_scaled, eps=0.75, min_samples=5)
print(f"DBSCAN formed clusters with {noise_cnt} noise points detected.")

### 8. Model Evaluation & Benchmarking

In [ ]:
results_map = {
    "K-Means": kmeans_labels,
    "Hierarchical": hier_labels,
    "DBSCAN": dbscan_labels
}

comp_table = ModelEvaluator.compare_models(X_scaled, results_map)
comp_table

### 9. Customer Segment Profiling & Business Intelligence

In [ ]:
analyzer = SegmentAnalyzer()
df_clean['Cluster'] = kmeans_labels

stats = analyzer.compute_cluster_stats(df_clean, kmeans_labels)
seg_names = analyzer.generate_segment_names(stats)
stats['Segment Name'] = stats['Cluster'].map(seg_names)

stats[['Cluster', 'Segment Name', 'Customer_Count', 'Customer_Percentage', 'Annual Income (k$)', 'Spending Score (1-100)', 'Purchase Frequency (per year)']]

### 10. Actionable Marketing Recommendations

In [ ]:
recs = analyzer.generate_business_recommendations(stats, seg_names)
for cid, val in recs.items():
    print(f"\n=== Cluster {cid}: {val['segment_name']} ===")
    for r in val['recommendations']:
        print(f"  * {r}")